In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
RESULT_PATH = Path("../models/absa/test_results.json")

with open(RESULT_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

rows = []

for model_key, model_data in results.items():
    model_name = model_data["model_name"]
    dataset_version = model_data["dataset_version"]

    for aspect, aspect_data in model_data["aspect_results"].items():
        metrics = aspect_data["metrics"]

        rows.append({
            "model": model_name,
            "dataset_version": dataset_version,
            "aspect": aspect,
            "accuracy": metrics["accuracy"],
            "precision_macro": metrics["precision_macro"],
            "recall_macro": metrics["recall_macro"],
            "f1_macro": metrics["f1_macro"]
        })

df = pd.DataFrame(rows)
print(df.head())

In [ ]:
avg_f1 = df.groupby(["model", "dataset_version"])["f1_macro"].mean().reset_index()
print("\nAverage F1:")
print(avg_f1)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=avg_f1, x="model", y="f1_macro", hue="dataset_version")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f")
plt.title("Average Macro F1 Score")
plt.ylabel("F1 Macro")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

In [ ]:
df["model_version"] = df["model"] + "_" + df["dataset_version"]
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df, x="aspect", y="f1_macro", hue="model_version")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f")
plt.title("F1 Macro Score by Aspect")
plt.ylabel("F1 Macro")
plt.xlabel("Aspect")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
pivot_df = df.pivot_table(index="aspect", columns="model_version", values="f1_macro")
plt.figure(figsize=(10, 5))
sns.heatmap(pivot_df, annot=True, fmt=".3f", cmap="Blues")
plt.title("F1 Macro Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
time_rows = []

for model_key, model_data in results.items():
    time_rows.append({
        "model": model_data["model_name"],
        "dataset_version": model_data["dataset_version"],
        "evaluation_time_seconds": model_data["evaluation_time_seconds"]
    })

time_df = pd.DataFrame(time_rows)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=time_df, x="model", y="evaluation_time_seconds", hue="dataset_version")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f")
plt.title("Evaluation Time Comparison")
plt.ylabel("Evaluation Time (seconds)")
plt.xlabel("Model")
plt.tight_layout()
plt.show()